# 🧠 Notebook 13: Distributed and Sharded Tensors

## 1. Purpose + Scope

This notebook covers large-scale tensor operations:

*   **Multi-Shard Tensor Logic**: Splitting huge tensors across nodes.
*   **Deterministic Distribution Model**: How shards are assigned to peers.
*   **Strict Mode Boundaries**: When distributed ops are allowed.

## 2. Spec References

*   `spec/t81-data-types.md`
*   `include/t81/tensor/distributed.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: The *results* of distributed computation must be deterministic, even if the execution timing is not.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Sharding Logic

Sharding is based on content hashing or explicit partitioning. We simulate a sharded matmul.

In [ ]:
# Conceptual: Split a 4x4 matrix into 4 2x2 shards
matrix_size = 4
shard_size = 2

def get_shard_id(row, col):
    shard_row = row // shard_size
    shard_col = col // shard_size
    return (shard_row, shard_col)

print(f"Element (0,0) -> Shard {get_shard_id(0,0)}")
print(f"Element (3,3) -> Shard {get_shard_id(3,3)}")

## 6. Deterministic Distribution

Nodes in the network agree on which peer holds which shard using a deterministic mapping (e.g., consistent hashing).

In [ ]:
import hashlib

def get_node_for_shard(shard_id, nodes):
    h = hashlib.sha256(str(shard_id).encode()).hexdigest()
    idx = int(h, 16) % len(nodes)
    return nodes[idx]

nodes = ["NodeA", "NodeB", "NodeC"]
shard = (0, 0)
assigned_node = get_node_for_shard(shard, nodes)
print(f"Shard {shard} assigned to {assigned_node}")

## 7. Strict Mode Boundaries

Operations that span shards must handle failures deterministically. If a node is down, the computation fails cleanly or uses redundant shards.

## 8. Architectural Commentary

Distributed tensors allow training models larger than any single GPU memory. T81's consensus ensures that even distributed training steps are verifiable.